# 01 - Data Understanding

## Objective

The objective of this notebook is to perform an initial inspection of the FAA Wildlife Strike dataset before any preprocessing.

The notebook focuses on:

- Understanding the dataset structure
- Inspecting data quality
- Identifying missing values
- Detecting duplicate records
- Understanding variable types
- Recording observations for the data preparation stage


## 1. Import Libraries

This section imports the libraries required for the initial inspection of the FAA Wildlife Strike dataset.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from pathlib import Path

## 2. Define Project Paths

The notebook is located inside the `notebooks` folder. Therefore, the project root is one directory above the current notebook location.

In [2]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw-data directory: {RAW_DATA_DIR}")
print(f"Processed-data directory: {PROCESSED_DATA_DIR}")

Project root: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis
Raw-data directory: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis\data\raw
Processed-data directory: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis\data\processed


## 3. Locate the Raw Dataset

Before reading the dataset, we confirm which files are available in the raw-data directory.

In [3]:
raw_files = list(RAW_DATA_DIR.glob("*"))

if not raw_files:
    print("No raw-data file was found. Add the FAA dataset to data/raw/.")
else:
    for file_path in raw_files:
        print(file_path.name)

.gitkeep
faa_strikes.csv


## 4. Load the Dataset

The dataset is loaded without applying any cleaning so that the original structure can be examined first.

In [4]:
DATA_FILE = RAW_DATA_DIR / "faa_strikes.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_FILE}\n"
        "Copy the FAA CSV file into the data/raw folder."
    )

df = pd.read_csv(
    DATA_FILE,
    encoding="latin-1",
    low_memory=False
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 5. Dataset Dimensions

The number of rows represents reported wildlife-strike records, while the number of columns represents the available variables.

In [5]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.info()
df.describe(include="all").T

Rows: 348,146
Columns: 103
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348146 entries, 0 to 348145
Columns: 103 entries, INDEX_NR to TRANSFER
dtypes: float64(18), int64(41), object(44)
memory usage: 273.6+ MB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
INDEX_NR,348146.0,NaN,NaN,NaN,1048104.275525,399139.760416,608242.0,707541.25,810386.5,1470809.75,1861554.0
INCIDENT_DATE,348146,13223,10/8/2025 0:00:00,197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
INCIDENT_MONTH,348146.0,NaN,NaN,NaN,7.190753,2.783914,1.0,5.0,8.0,9.0,12.0
INCIDENT_YEAR,348146.0,NaN,NaN,NaN,2013.866002,9.056416,1990.0,2008.0,2016.0,2022.0,2026.0
TIME,226783,1441,,28297,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
REPORTED_TITLE,348146,1,REDACTED,348146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SOURCE,348146,16,FAA Form 5200-7-E,216344,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PERSON,327040,6,Carcass Found,95738,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LUPDATE,348146,4834,5/20/2024 0:00:00,10598,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Sample Records

A small sample is displayed to understand the structure and formatting of the original records.

In [6]:
df.head()

,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,AIRPORT_LATITUDE,AIRPORT_LONGITUDE,...,NR_INJURIES,NR_FATALITIES,COMMENTS,IMAGE,REPORTED_NAME,REPORTED_TITLE,SOURCE,PERSON,LUPDATE,TRANSFER
0,608242,6/22/1996 0:00:00,6,1996,NaN,NaN,KSMF,SACRAMENTO INTL,NaN,NaN,...,NaN,NaN,/Legacy Record 100001/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0
1,608243,6/26/1996 0:00:00,6,1996,NaN,NaN,KDEN,DENVER INTL AIRPORT,NaN,NaN,...,NaN,NaN,/Legacy Record 100002/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0
2,608244,7/1/1996 0:00:00,7,1996,NaN,NaN,KOMA,EPPLEY AIRFIELD,NaN,NaN,...,NaN,NaN,/Legacy Record 100003/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0
3,608245,7/1/1996 0:00:00,7,1996,NaN,NaN,KIAD,WASHINGTON DULLES INTL ARPT,NaN,NaN,...,NaN,NaN,/Legacy Record 100004/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0
4,608246,7/1/1996 0:00:00,7,1996,NaN,NaN,KLGA,LA GUARDIA ARPT,NaN,NaN,...,NaN,NaN,/Legacy Record 100005/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0


## 7. Columns and Data Types

This inspection identifies numeric, text, date-like, and categorical variables that may require different preparation steps.

In [7]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "unique_values": df.nunique(dropna=True).values,
})

column_summary

,column,dtype,non_null_count,unique_values
0,INDEX_NR,int64,348146,348139
1,INCIDENT_DATE,object,348146,13223
2,INCIDENT_MONTH,int64,348146,12
3,INCIDENT_YEAR,int64,348146,37
4,TIME,object,226783,1441
...,...,...,...,...
98,REPORTED_TITLE,object,348146,1
99,SOURCE,object,348146,16
100,PERSON,object,327040,6
101,LUPDATE,object,348146,4834


## 8. Missing-Value Overview

Missingness is measured before cleaning to identify variables that may require exclusion, imputation, grouping, or special interpretation.

In [8]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(
          missing_percent=lambda x:
              (x["missing_count"] / len(df) * 100).round(2)
      )
      .sort_values("missing_percent", ascending=False)
)

missing_summary.head(20)

,missing_count,missing_percent
INCIDENT_LATITUDE,348146,100.00
INCIDENT_LONGITUDE,348146,100.00
AIRPORT_LATITUDE,348146,100.00
AIRPORT_LONGITUDE,348146,100.00
NR_FATALITIES,348121,99.99
NR_INJURIES,347844,99.91
BIRD_BAND_NUMBER,347352,99.77
EFFECT_OTHER,345363,99.20
ENG_4_POS,344505,98.95
COST_REPAIRS,342721,98.44


## 9. Exact Duplicate Records

Exact duplicates are inspected but are not removed at this stage because this notebook focuses on understanding the original dataset.

In [9]:
exact_duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicate_count:,}")
print(
    f"Duplicate percentage: "
    f"{exact_duplicate_count / len(df) * 100:.2f}%"
)

Exact duplicate rows: 0
Duplicate percentage: 0.00%


## 10. Primary Target Variable Analysis

The primary modelling objective of this project is to predict whether an aircraft sustained damage after a wildlife strike.

This section examines the primary target variable, including its categories, missing values, and class distribution.

In [10]:
damage_columns = [
    col
    for col in df.columns
    if "DAMAGE" in col.upper()
]

damage_columns

['INDICATED_DAMAGE', 'DAMAGE_LEVEL']

In [11]:
PRIMARY_TARGET = "INDICATED_DAMAGE"

target_counts = df[PRIMARY_TARGET].value_counts(dropna=False)

target_percentages = (
    df[PRIMARY_TARGET]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percentages
})

target_summary

,Count,Percentage
INDICATED_DAMAGE,,
0,326100,93.67
1,22046,6.33


In [12]:
print(f"Target column: {PRIMARY_TARGET}")
print(f"Unique classes: {df[PRIMARY_TARGET].nunique()}")
print(f"Missing values: {df[PRIMARY_TARGET].isna().sum()}")
print(f"Missing percentage: {df[PRIMARY_TARGET].isna().mean()*100:.2f}%")

Target column: INDICATED_DAMAGE
Unique classes: 2
Missing values: 0
Missing percentage: 0.00%


### Initial Findings

The primary target variable (`INDICATED_DAMAGE`) is a binary variable with two classes (0 and 1).

No missing values were identified in the target variable.

The class distribution is highly imbalanced:

- Class 0 (No indicated damage): **93.67%**
- Class 1 (Indicated damage): **6.33%**

This imbalance should be considered during model development and evaluation. Performance metrics beyond overall accuracy (such as Precision, Recall, F1-score, ROC-AUC, and PR-AUC) may be more appropriate.

## 11. Secondary Damage Variables

In addition to the primary target, the dataset contains several damage-related variables that may support descriptive analysis or future modelling tasks.

This section examines their completeness and basic characteristics.

In [13]:
damage_columns

['INDICATED_DAMAGE', 'DAMAGE_LEVEL']

In [14]:
secondary_damage_summary = pd.DataFrame({
    "Column": damage_columns,
    "Data Type": [str(df[col].dtype) for col in damage_columns],
    "Missing Count": [df[col].isna().sum() for col in damage_columns],
    "Missing %": [
        round(df[col].isna().mean() * 100, 2)
        for col in damage_columns
    ],
    "Unique Values": [
        df[col].nunique(dropna=True)
        for col in damage_columns
    ]
})

secondary_damage_summary

,Column,Data Type,Missing Count,Missing %,Unique Values
0,INDICATED_DAMAGE,int64,0,0.00,2
1,DAMAGE_LEVEL,object,122605,35.22,5


In [15]:
damage_level_summary = (
    df["DAMAGE_LEVEL"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

damage_level_summary["Percentage"] = (
    df["DAMAGE_LEVEL"]
    .value_counts(dropna=False, normalize=True)
    * 100
).round(2)

damage_level_summary

,Count,Percentage
DAMAGE_LEVEL,,
N,203496,58.45
NaN,122605,35.22
M,8783,2.52
M?,8772,2.52
S,4401,1.26
D,89,0.03


### Initial Findings

The `DAMAGE_LEVEL` variable contains five recorded damage categories together with a substantial proportion of missing values (35.22%).

Among the available records:

- `N` (No Damage) is the dominant category (58.45%).
- `M` and `M?` each account for approximately 2.5%.
- `S` (Substantial Damage) represents only 1.26%.
- `D` (Destroyed) is extremely rare (0.03%).

These results indicate that `DAMAGE_LEVEL` is not suitable as the primary prediction target because of its high missingness and severe class imbalance. However, it may still be useful for descriptive analysis or as a secondary outcome variable after appropriate preprocessing.

## 12. Repeated `INDEX_NR` Analysis

Although no exact duplicate rows were found, the dataset contains repeated `INDEX_NR` values.

This section investigates how many repeated identifiers exist and whether the associated records are identical or contain different information.

In [16]:
ID_COLUMN = "INDEX_NR"

print(f"Total rows: {len(df):,}")
print(f"Unique IDs: {df[ID_COLUMN].nunique():,}")
print(f"Repeated IDs: {len(df) - df[ID_COLUMN].nunique():,}")

Total rows: 348,146
Unique IDs: 348,139
Repeated IDs: 7


In [17]:
id_counts = df[ID_COLUMN].value_counts()

repeated_ids = id_counts[id_counts > 1]

print(f"Number of repeated INDEX_NR values: {len(repeated_ids)}")

repeated_ids

Number of repeated INDEX_NR values: 7


INDEX_NR
1850863    2
793166     2
814615     2
1850036    2
1859904    2
778860     2
1520761    2
Name: count, dtype: int64

In [18]:
# Display all records with repeated INDEX_NR values
repeated_records = (
    df[df[ID_COLUMN].isin(repeated_ids.index)]
    .sort_values(ID_COLUMN)
)

repeated_records

,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,AIRPORT_LATITUDE,AIRPORT_LONGITUDE,...,NR_INJURIES,NR_FATALITIES,COMMENTS,IMAGE,REPORTED_NAME,REPORTED_TITLE,SOURCE,PERSON,LUPDATE,TRANSFER
152380,778860,3/1/2016 0:00:00,3,2016,NaN,NaN,KPSM,PORTSMOUTH INTL ARPT AT PEASE,NaN,NaN,...,NaN,NaN,2016-3-1-211917 /Legacy Record 371717/,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,9/1/2016 0:00:00,0
152381,778860,3/1/2016 0:00:00,3,2016,NaN,NaN,KPSM,PORTSMOUTH INTL ARPT AT PEASE,NaN,NaN,...,NaN,NaN,2016-3-1-211917 /Legacy Record 371717/,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,9/1/2016 0:00:00,0
162301,793166,4/29/2017 0:00:00,4,2017,NaN,NaN,KNUQ,MOFFETT FEDERAL AIRFIELD,NaN,NaN,...,NaN,NaN,2017-6-12-173440 /Legacy Record 390497/,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,5/18/2018 0:00:00,0
162302,793166,4/29/2017 0:00:00,4,2017,NaN,NaN,KNUQ,MOFFETT FEDERAL AIRFIELD,NaN,NaN,...,NaN,NaN,2017-6-12-173440 /Legacy Record 390497/,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,5/18/2018 0:00:00,0
177357,814615,9/27/2018 0:00:00,9,2018,NaN,NaN,KBDL,BRADLEY INTL,NaN,NaN,...,NaN,NaN,2018-9-28-111208 /Legacy Record 416060/,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,11/27/2018 0:00:00,0
177358,814615,9/27/2018 0:00:00,9,2018,NaN,NaN,KBDL,BRADLEY INTL,NaN,NaN,...,NaN,NaN,2018-9-28-111208 /Legacy Record 416060/,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,11/27/2018 0:00:00,0
278303,1520761,3/4/2017 0:00:00,3,2017,NaN,NaN,KISN,SLOULIN FLD INTL ARPT,NaN,NaN,...,NaN,NaN,NaN,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,5/15/2024 0:00:00,0
278304,1520761,3/4/2017 0:00:00,3,2017,NaN,NaN,KISN,SLOULIN FLD INTL ARPT,NaN,NaN,...,NaN,NaN,NaN,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,5/15/2024 0:00:00,0
344180,1850036,2/13/2026 0:00:00,2,2026,,NaN,KPAO,PALO ALTO ARPT OF SANTA CLARA COUNTY,NaN,NaN,...,NaN,NaN,revised species,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,5/29/2026 0:00:00,0
344181,1850036,2/13/2026 0:00:00,2,2026,,NaN,KPAO,PALO ALTO ARPT OF SANTA CLARA COUNTY,NaN,NaN,...,NaN,NaN,revised species,0,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,5/29/2026 0:00:00,0


In [19]:
# Check whether repeated INDEX_NR records are identical
for record_id in repeated_ids.index:
    group = df[df[ID_COLUMN] == record_id]

    # Count columns with more than one distinct value
    different_columns = [
        column
        for column in df.columns
        if group[column].nunique(dropna=False) > 1
    ]

    print(f"\nINDEX_NR: {record_id}")

    if len(different_columns) == 0:
        print("✓ All values are identical across duplicated records.")
    else:
        print("Different columns:")
        print(different_columns)


INDEX_NR: 1850863
Different columns:
['SPECIES']

INDEX_NR: 793166
Different columns:
['SPECIES']

INDEX_NR: 814615
Different columns:
['SPECIES']

INDEX_NR: 1850036
Different columns:
['SPECIES']

INDEX_NR: 1859904
Different columns:
['SPECIES']

INDEX_NR: 778860
Different columns:
['SPECIES']

INDEX_NR: 1520761
Different columns:
['SPECIES']


### Initial Findings

The dataset contains only **7 repeated `INDEX_NR` values** out of **348,146** total records.

Inspection of these repeated identifiers shows that the duplicated records differ only in the **`SPECIES`** field, while all other variables remain identical.

This suggests that these records may represent revisions or updates to species identification rather than accidental duplicate entries.

Because repeated identifiers account for only a very small proportion of the dataset and are not exact duplicates, **no records are removed during the Data Understanding stage**. Their treatment will be reviewed further during Data Preparation.

## 13. Temporal Coverage

This section examines the temporal coverage of the dataset, including the range of years and the number of records reported each year.

Understanding the time distribution is important because the project plans to use chronological validation when developing predictive models.

In [20]:
YEAR_COLUMN = "INCIDENT_YEAR"

print(f"Earliest year: {df[YEAR_COLUMN].min()}")
print(f"Latest year: {df[YEAR_COLUMN].max()}")

print(f"\nNumber of unique years: {df[YEAR_COLUMN].nunique()}")

Earliest year: 1990
Latest year: 2026

Number of unique years: 37


In [21]:
records_per_year = (
    df[YEAR_COLUMN]
    .value_counts()
    .sort_index()
    .to_frame("Record Count")
)

records_per_year

,Record Count
INCIDENT_YEAR,
1990,2120
1991,2515
1992,2652
1993,2625
1994,2707
1995,2827
1996,3031
1997,3560
1998,3808


In [23]:
df[df["INCIDENT_YEAR"] == 2026]["INCIDENT_DATE"].describe()

count                  4576
unique                  132
top       4/24/2026 0:00:00
freq                    105
Name: INCIDENT_DATE, dtype: object

In [25]:
incident_2026 = df[df["INCIDENT_YEAR"] == 2026]
incident_2026["INCIDENT_DATE"].tail()

348141    5/1/2026 0:00:00
348142    5/1/2026 0:00:00
348143    5/1/2026 0:00:00
348144    5/3/2026 0:00:00
348145    5/3/2026 0:00:00
Name: INCIDENT_DATE, dtype: object

### Initial Findings

The dataset spans **37 years**, covering wildlife-strike records from **1990 to 2026**.

The annual number of reported incidents generally increases over time, indicating broader reporting coverage and a growing number of recorded wildlife-strike events in recent years.

A noticeable decline in reported incidents is observed in **2020**, followed by recovery in subsequent years.

Although records are available for **2026**, only **4,576** incidents are currently included, and the latest available records extend only to **early May 2026**. This indicates that **2026 is an incomplete reporting year** rather than a full calendar year.

Consequently, special consideration should be given when performing chronological train/validation/test splits, as using the incomplete 2026 data without adjustment may introduce bias.

# 14. Initial Observations

## Summary

Based on the exploratory analysis, the following observations were identified:

- The primary target variable (`INDICATED_DAMAGE`) is complete (no missing values) but exhibits significant class imbalance (93.67% vs. 6.33%).
- The secondary damage variable (`DAMAGE_LEVEL`) contains approximately 35% missing values and highly imbalanced severity categories.
- Only 7 repeated `INDEX_NR` values were found among 348,146 records. These records differ only in the `SPECIES` field, suggesting species revisions rather than accidental duplicate records.
- The dataset covers wildlife-strike incidents from **1990 to 2026**. However, records for **2026** currently extend only to early May, indicating that it is an incomplete reporting year.
- Missing values are concentrated in specific variables and will be addressed during the data preparation stage.
- The dataset contains numerical, categorical, and temporal variables suitable for further preprocessing and predictive modelling.

The next notebook (**02_data_preparation.ipynb**) will focus on data cleaning, preprocessing, feature selection, and preparation for modelling.